<a href="https://colab.research.google.com/github/rajivagarwal221/Rocket-Land/blob/main/notebooks/v29_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V29 — classical guidance + residual RL (3D rocket landing)
Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**. Every run resumes from `MyDrive/rocket_v29/`.

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/rocket_v29'
os.makedirs(f'{DRIVE}/logs', exist_ok=True)
%pip install -q mujoco==4.0.1 gymnasium==1.4.0 stable-baselines3==2.9.2 tensorboard
%pip uninstall -y -q tensorflow tensorflow-cpu
if not os.path.isdir('/content/rocket'):
    !git clone --depth 1 https://github.com/rajivagarwal221/Rocket-Land.git /content/rocket
else:
    !cd /content/rocket && git fetch --depth 1 origin main -q && git reset --hard origin/main -q
%cd /content/rocket
!git log --oneline -1
!python -m unittest discover -s tests 2>&1 | tail -3

## 1. Classical baseline audit — 1000 held-out seeds, every factor on

In [ ]:
!python src/evaluate_v29.py --episodes 1000 --seed 90000 --difficulty 1 --delay 2 --noise --dropout 0.005 --out {DRIVE}/logs/classical_heldout.csv | tail -12

## 2. Train the residual (auto-resumes; re-run this cell after any disconnect)

In [ ]:
import os
import subprocess
import sys

subprocess.run(
    [sys.executable, "-u", "src/train_v29_residual.py"],
    cwd="/content/rocket",
    env={
        **os.environ,
        "V29_OUT": "/content/drive/MyDrive/rocket_v29",
        "V29_STEPS": "2000000",
        "V29_NENVS": "8",
        "V29_FRESH": "0",
    },
    check=True,
)

## 3. Audit the best residual on the same 1000 held-out seeds

In [ ]:
BEST = f'{DRIVE}/models/master_v29_residual_best.zip'
if os.path.isfile(BEST):
    !python src/evaluate_v29.py --model {BEST} --episodes 1000 --seed 90000 --difficulty 1 --delay 2 --noise --dropout 0.005 --out {DRIVE}/logs/residual_heldout.csv | tail -12
else:
    print('No residual has beaten the classical baseline yet.')